In [1]:
# 0. prepare
# We have downloaded Cam-CAN dataset in our computer, with structure: D:/data/CamCan_single_shell/{subject}/Preprocessed_data/.
# We will run "0_prepare.sh" in our environment

import sys
import subprocess

ROOT = "D:/data/CamCan_single_shell"

cmd = [
    sys.executable, "-m", "dti_shnet.preprocess",

    "--root", ROOT,
    "--layout-preset", "camcan",
    "--out-dir-template", "preprocessed_test/{subject}",

    "--source-nii-template", "{subject}/Preprocessed_data/dwi_preprocessed_single.nii.gz",
    "--source-bval-template", "{subject}/Preprocessed_data/dwi_preprocessed_single.bval",
    "--source-bvec-template", "{subject}/Preprocessed_data/dwi_preprocessed_single.bvec",

    "--target-nii-template", "{subject}/Preprocessed_data/dwi_preprocessed.nii.gz",
    "--target-bval-template", "{subject}/Preprocessed_data/dwi_preprocessed.bval",
    "--target-bvec-template", "{subject}/Preprocessed_data/dwi_preprocessed.bvec",

    "--workers", "8",
    "--b-in", "1000",
    "--b-out", "2000",
    "--b-tol", "50",
    "--st-clip-pct", "99.5",
    "--lmax", "8",
    "--lam", "0.006",
]

print("[python]", sys.executable)
print("[command]")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

[python] C:\Users\nkvhu\miniconda3\python.exe
[command]
C:\Users\nkvhu\miniconda3\python.exe -m dti_shnet.preprocess --root D:/data/CamCan_single_shell --layout-preset camcan --out-dir-template preprocessed_test/{subject} --source-nii-template {subject}/Preprocessed_data/dwi_preprocessed_single.nii.gz --source-bval-template {subject}/Preprocessed_data/dwi_preprocessed_single.bval --source-bvec-template {subject}/Preprocessed_data/dwi_preprocessed_single.bvec --target-nii-template {subject}/Preprocessed_data/dwi_preprocessed.nii.gz --target-bval-template {subject}/Preprocessed_data/dwi_preprocessed.bval --target-bvec-template {subject}/Preprocessed_data/dwi_preprocessed.bvec --workers 8 --b-in 1000 --b-out 2000 --b-tol 50 --st-clip-pct 99.5 --lmax 8 --lam 0.006


CompletedProcess(args=['C:\\Users\\nkvhu\\miniconda3\\python.exe', '-m', 'dti_shnet.preprocess', '--root', 'D:/data/CamCan_single_shell', '--layout-preset', 'camcan', '--out-dir-template', 'preprocessed_test/{subject}', '--source-nii-template', '{subject}/Preprocessed_data/dwi_preprocessed_single.nii.gz', '--source-bval-template', '{subject}/Preprocessed_data/dwi_preprocessed_single.bval', '--source-bvec-template', '{subject}/Preprocessed_data/dwi_preprocessed_single.bvec', '--target-nii-template', '{subject}/Preprocessed_data/dwi_preprocessed.nii.gz', '--target-bval-template', '{subject}/Preprocessed_data/dwi_preprocessed.bval', '--target-bvec-template', '{subject}/Preprocessed_data/dwi_preprocessed.bvec', '--workers', '8', '--b-in', '1000', '--b-out', '2000', '--b-tol', '50', '--st-clip-pct', '99.5', '--lmax', '8', '--lam', '0.006'], returncode=0)

In [2]:
# 1. train
# Now, we have prepared the data and have saved them in D:/data/CamCan_single_shell/preprocessed_test/{subject}.
# Under every directory for each subject, we have:
# (1) source/target St signals from single-/multi-shell dMRI
# (2) FA/MD/brain_mask for DTI conditions 
# (3) SH matric shmeta_b1000_b2000_l8_lam0.006
# (4) Extrapolation baseline dwi_dti_baseline and signal_dti_baseline, to pad the voxel outside the ROI regions
# (5) S0 to restore the raw dMRI from St signal
# (6) other information: source_to_full_map to mark which slice is target b (e.g. 2000), and the bval and bvec files
# Here, we need to train our DTI-SHNet using the preprocessed data

import sys
import subprocess
from pathlib import Path

# ========= config =========
PREPROC_ROOT = "D:/data/CamCan_single_shell/preprocessed_test"
OUT_DIR = "./results"
EVAL_ID_FILE = "splits/eval_id.txt"

DTI = 1
SIG = 1

# ========= optional checks =========
print("[python]", sys.executable)
print("[preproc root]", PREPROC_ROOT)
print("[out dir]", OUT_DIR)
print("[eval id file]", EVAL_ID_FILE)

if not Path(PREPROC_ROOT).exists():
    raise FileNotFoundError(f"PREPROC_ROOT not found: {PREPROC_ROOT}")

if not Path(EVAL_ID_FILE).exists():
    raise FileNotFoundError(
        f"EVAL_ID_FILE not found: {EVAL_ID_FILE}\n"
        "Make sure the notebook current directory is the project root."
    )

# ========= command =========
cmd = [
    sys.executable, "-m", "dti_shnet.train",

    "--roots", repr([PREPROC_ROOT]),
    "--out-dir", OUT_DIR,
    "--eval-id-file", EVAL_ID_FILE,

    "--use-dti", str(DTI),
    "--use-sig-loss", str(SIG),
    "--lambda-sig", "0.1",

    "--roi", "96,96,64",
    "--patch", "32,32,32",
    "--dim", "64",
    "--batch", "4",
    "--epochs", "30",
    "--lr", "2e-4",
    "--num-workers", "8",

    "--pps-train", "12",
    "--pps-val", "1",
    "--shuffle-patches", "0",
    "--seed-per-epoch", "1",

    "--amp", "1",
    "--seed", "0",

    "--b-in", "1000",
    "--b-out", "2000",
    "--lmax", "8",
    "--lam", "0.006",
    "--canon-n", "60",
]

print("[command]")
print(" ".join(cmd))

subprocess.run(cmd, check=True)


[python] C:\Users\nkvhu\miniconda3\python.exe
[preproc root] D:/data/CamCan_single_shell/preprocessed_test
[out dir] ./results
[eval id file] splits/eval_id.txt
[command]
C:\Users\nkvhu\miniconda3\python.exe -m dti_shnet.train --roots ['D:/data/CamCan_single_shell/preprocessed_test'] --out-dir ./results --eval-id-file splits/eval_id.txt --use-dti 1 --use-sig-loss 1 --lambda-sig 0.1 --roi 96,96,64 --patch 32,32,32 --dim 64 --batch 4 --epochs 1 --lr 2e-4 --num-workers 8 --pps-train 12 --pps-val 1 --shuffle-patches 0 --seed-per-epoch 1 --amp 1 --seed 0 --b-in 1000 --b-out 2000 --lmax 8 --lam 0.006 --canon-n 60


CompletedProcess(args=['C:\\Users\\nkvhu\\miniconda3\\python.exe', '-m', 'dti_shnet.train', '--roots', "['D:/data/CamCan_single_shell/preprocessed_test']", '--out-dir', './results', '--eval-id-file', 'splits/eval_id.txt', '--use-dti', '1', '--use-sig-loss', '1', '--lambda-sig', '0.1', '--roi', '96,96,64', '--patch', '32,32,32', '--dim', '64', '--batch', '4', '--epochs', '1', '--lr', '2e-4', '--num-workers', '8', '--pps-train', '12', '--pps-val', '1', '--shuffle-patches', '0', '--seed-per-epoch', '1', '--amp', '1', '--seed', '0', '--b-in', '1000', '--b-out', '2000', '--lmax', '8', '--lam', '0.006', '--canon-n', '60'], returncode=0)

In [3]:
# 2. test
# Now, we finish training our DTI-SHNet and save the checkpoint in ./results/last.pt or ./results/best.pt. 
# We will use last.pt to generate St signal for test set.

import sys
import subprocess
from pathlib import Path

# ========= config =========
PREPROC_ROOT = "D:/data/CamCan_single_shell/preprocessed_test"
CKPT = "./results/last.pt"
SAVE_DIR = "./results"
EVAL_ID_FILE = "splits/eval_id.txt"

# ========= optional checks =========
print("[python]", sys.executable)
print("[preproc root]", PREPROC_ROOT)
print("[ckpt]", CKPT)
print("[save dir]", SAVE_DIR)
print("[eval id file]", EVAL_ID_FILE)

if not Path(PREPROC_ROOT).exists():
    raise FileNotFoundError(f"PREPROC_ROOT not found: {PREPROC_ROOT}")

if not Path(CKPT).exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT}")

if not Path(EVAL_ID_FILE).exists():
    raise FileNotFoundError(
        f"EVAL_ID_FILE not found: {EVAL_ID_FILE}\n"
        "Make sure the notebook current directory is the project root."
    )

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# ========= command =========
cmd = [
    sys.executable, "-m", "dti_shnet.infer",

    "--roots", repr([PREPROC_ROOT]),
    "--ckpt", CKPT,
    "--save-dir", SAVE_DIR,
    "--eval-id-file", EVAL_ID_FILE,

    "--roi", "96,96,64",
    "--infer-patch", "32,32,32",
    "--overlap", "0.5",

    "--use-dti", "1",

    "--b-in", "1000",
    "--b-out", "2000",
    "--lmax", "8",
    "--lam", "0.006",
]

print("[command]")
print(" ".join(cmd))

subprocess.run(cmd, check=True)

[python] C:\Users\nkvhu\miniconda3\python.exe
[preproc root] D:/data/CamCan_single_shell/preprocessed_test
[ckpt] ./results/last.pt
[save dir] ./results
[eval id file] splits/eval_id.txt
[command]
C:\Users\nkvhu\miniconda3\python.exe -m dti_shnet.infer --roots ['D:/data/CamCan_single_shell/preprocessed_test'] --ckpt ./results/last.pt --save-dir ./results --eval-id-file splits/eval_id.txt --roi 96,96,64 --infer-patch 32,32,32 --overlap 0.5 --use-dti 1 --b-in 1000 --b-out 2000 --lmax 8 --lam 0.006


CompletedProcess(args=['C:\\Users\\nkvhu\\miniconda3\\python.exe', '-m', 'dti_shnet.infer', '--roots', "['D:/data/CamCan_single_shell/preprocessed_test']", '--ckpt', './results/last.pt', '--save-dir', './results', '--eval-id-file', 'splits/eval_id.txt', '--roi', '96,96,64', '--infer-patch', '32,32,32', '--overlap', '0.5', '--use-dti', '1', '--b-in', '1000', '--b-out', '2000', '--lmax', '8', '--lam', '0.006'], returncode=0)

In [4]:
# 3. evaluation
# We have generated the artificial St signals and have saved them at ./results/dti_shnet_pred/{subject}/signal_pred.nii.gz.
# Now, we will calculate the visual metrics, DTI errors, NODDI errors, and then generate the S images of raw domain by S = St * S0

import sys
import subprocess
from pathlib import Path

# ========= config =========
LAYOUT = "camcan"

ORIG_ROOT = "D:/data/CamCan_single_shell"
PREPROC_ROOT = "D:/data/CamCan_single_shell/preprocessed_test"
RUN_DIR = "./results"

PRED_ROOT = f"{RUN_DIR}/dti_shnet_pred"
EXPORT_ROOT = f"{RUN_DIR}/export_dMRI"
DTI_OUT = f"{RUN_DIR}/dti"
NODDI_OUT = f"{RUN_DIR}/noddi"

B_IN = "1000"
B_OUT = "2000"
LMAX = "8"
LAM = "0.006"
TARGET_B = "2000"
B_TOL = "50"
ST_CLIP_PCT = "99.5"

VIS_WORKERS = "4"
EXPORT_WORKERS = "4"
DTI_WORKERS = "4"

# NODDI is slow and requires dmri-amico + MRtrix.
RUN_NODDI = "1"
NODDI_WORKERS = "4"
NODDI_THREADS = "8"

# ========= checks =========
print("[python]", sys.executable)
print("[config] LAYOUT =", LAYOUT)
print("[config] ORIG_ROOT =", ORIG_ROOT)
print("[config] PREPROC_ROOT =", PREPROC_ROOT)
print("[config] RUN_DIR =", RUN_DIR)
print("[config] PRED_ROOT =", PRED_ROOT)
print("[config] EXPORT_ROOT =", EXPORT_ROOT)
print("[config] DTI_OUT =", DTI_OUT)
print("[config] NODDI_OUT =", NODDI_OUT)

if not Path(ORIG_ROOT).exists():
    raise FileNotFoundError(f"ORIG_ROOT not found: {ORIG_ROOT}")

if not Path(PREPROC_ROOT).exists():
    raise FileNotFoundError(f"PREPROC_ROOT not found: {PREPROC_ROOT}")

if not Path(PRED_ROOT).exists():
    raise FileNotFoundError(
        f"PRED_ROOT not found: {PRED_ROOT}\n"
        "Please make sure inference has already generated predictions there."
    )

Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
Path(EXPORT_ROOT).mkdir(parents=True, exist_ok=True)
Path(DTI_OUT).mkdir(parents=True, exist_ok=True)
Path(NODDI_OUT).mkdir(parents=True, exist_ok=True)


def run_cmd(cmd):
    print("\n[command]")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


# ========= 1. visual metrics =========
cmd_visual = [
    sys.executable, "-m", "dti_shnet.evaluation.visual",

    "--pred-root", PRED_ROOT,
    "--preproc-root", PREPROC_ROOT,
    "--out-csv", f"{RUN_DIR}/visual_metrics.csv",

    "--b-in", B_IN,
    "--b-out", B_OUT,
    "--lmax", LMAX,
    "--lam", LAM,

    "--num-workers", VIS_WORKERS,
]

run_cmd(cmd_visual)


# ========= 2. export predicted dMRI =========
cmd_export = [
    sys.executable, "-m", "dti_shnet.export_dwi",

    "--layout-preset", LAYOUT,
    "--orig-root", ORIG_ROOT,
    "--preproc-root", PREPROC_ROOT,
    "--pred-root", PRED_ROOT,
    "--out-root", EXPORT_ROOT,

    "--target-b", TARGET_B,
    "--b-tol", B_TOL,
    "--st-clip-pct", ST_CLIP_PCT,

    "--num-workers", EXPORT_WORKERS,
    "--executor", "process",
    "--skip-existing",
]

run_cmd(cmd_export)


# ========= 3. DTI evaluation =========
cmd_dti = [
    sys.executable, "-m", "dti_shnet.evaluation.dti",

    "--layout-preset", LAYOUT,
    "--orig-root", ORIG_ROOT,
    "--preproc-root", PREPROC_ROOT,
    "--pred-root", PRED_ROOT,
    "--out-dir", DTI_OUT,

    "--target-b", TARGET_B,
    "--b-tol", B_TOL,
    "--st-clip-pct", ST_CLIP_PCT,

    "--mask-source", "mrtrix",
    "--mrtrix-clean-scale", "2",
    "--mrtrix-nthreads", "0",
    "--mrtrix-force", "0",

    "--dti-use-all-dw", "1",
    "--b1", B_IN,

    "--num-workers", DTI_WORKERS,
]

run_cmd(cmd_dti)


# ========= 4. optional NODDI evaluation =========
if RUN_NODDI == "1":
    cmd_noddi = [
        sys.executable, "-m", "dti_shnet.evaluation.noddi",

        "--layout-preset", LAYOUT,
        "--orig-root", ORIG_ROOT,
        "--preproc-root", PREPROC_ROOT,
        "--pred-root", PRED_ROOT,
        "--out-dir", NODDI_OUT,

        "--target-b", TARGET_B,
        "--b-tol", B_TOL,
        "--st-clip-pct", ST_CLIP_PCT,

        "--mask-source", "mrtrix",
        "--mrtrix-clean-scale", "2",
        "--mrtrix-nthreads", "0",
        "--mrtrix-force", "0",

        "--nb-threads", NODDI_THREADS,
        "--num-workers", NODDI_WORKERS,
    ]

    run_cmd(cmd_noddi)

[python] C:\Users\nkvhu\miniconda3\python.exe
[config] LAYOUT = camcan
[config] ORIG_ROOT = D:/data/CamCan_single_shell
[config] PREPROC_ROOT = D:/data/CamCan_single_shell/preprocessed_test
[config] RUN_DIR = ./results
[config] PRED_ROOT = ./results/dti_shnet_pred
[config] EXPORT_ROOT = ./results/export_dMRI
[config] DTI_OUT = ./results/dti
[config] NODDI_OUT = ./results/noddi

[command]
C:\Users\nkvhu\miniconda3\python.exe -m dti_shnet.evaluation.visual --pred-root ./results/dti_shnet_pred --preproc-root D:/data/CamCan_single_shell/preprocessed_test --out-csv ./results/visual_metrics.csv --b-in 1000 --b-out 2000 --lmax 8 --lam 0.006 --num-workers 4

[command]
C:\Users\nkvhu\miniconda3\python.exe -m dti_shnet.export_dwi --layout-preset camcan --orig-root D:/data/CamCan_single_shell --preproc-root D:/data/CamCan_single_shell/preprocessed_test --pred-root ./results/dti_shnet_pred --out-root ./results/export_dMRI --target-b 2000 --b-tol 50 --st-clip-pct 99.5 --num-workers 4 --executor pro